<a href="https://colab.research.google.com/github/rene-aum/Hermes/blob/Moises/Asignacion/nb2_asignacion_creditos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Automatización del proceso de Asignaciones
Recuerda que para este punto ya debiste subir la **base de clientes con Corte 1** del día.
Además, por favor:
1. Genera la hoja de sheets de salida en la carpeta correspondiente: https://drive.google.com/drive/folders/1AZuXU0aAEyzaA6OQbI_5F74Q568EoY-d

2. Actualiza con cuidado el **día** 📅
3. El número de **corte diario** 🔢  
4. La cosecha que estás asignando.

In [1]:
from datetime import datetime, timedelta

id_drive_apisFolios = '1OW4yxE7h8BCcn0mqhCfk05B4_27Ghvfd'
id_drive_ctes = '1UQvGtFjCLp47JrP7vuBRQpg9cfSAlJxH'
id_drive_hist = '1zvW-Dxow9gz1Dnbpg_jO7my4wadDvJDW'
id_drive_pedidos = '1yVMEVT9zooZXOsiYHnZ1FZVGDZJQQ-sv'
id_drive_catalogos = '1xQkepXoIoEHNgdYUaLAT7FtDnuzYo1qb'
id_drive_salidas = '1WvDfMwtbAqYSf1dJqHH_t6WbrFYE-1HC'
id_sheets_tc2 = '1k8rguLeF1O33XCaVDxPiQ1C4SbxLDSIeqNcriYtsF-k'

fh_salida = '2026-02-17' #@param{type:'date'}
fh_salida_dt = datetime.strptime(fh_salida, '%Y-%m-%d')
dia_salida = str(fh_salida_dt.day).zfill(2)
mes_salida = str(fh_salida_dt.month).zfill(2)
anio_salida = str(fh_salida_dt.year).zfill(4)
fh_de_asignacion = fh_salida_dt.strftime('%d-%m-%Y')

cosecha = 'Cosecha Mar 26' #@param{type:'string'}

nb_carpeta_ctes_mes = f'{anio_salida}{mes_salida}'
nb_ctes_csv = f'report_{anio_salida}{mes_salida}{dia_salida}_c1.csv'
nb_sheet_salida = f'Salidas {fh_salida}'

dicc_espacios = {'Reforma 510':'torre','MetrÃ³poli Patriotismo':'patriotismo','Samara SatÃ©lite':'samara'}
dicc_espacios2 = {'MetrÃ³poli Patriotismo': 'Metrópoli Patriotismo','Samara SatÃ©lite': 'Samara Satélite'}
dicc_espacios3 = {'torre':'Reforma 510','patriotismo':'Metrópoli Patriotismo','samara':'Samara Satélite'}

actualizar_tc = 'S' #@param{type:'string'}['S','N']

In [2]:
dia_salida, mes_salida, anio_salida, nb_carpeta_ctes_mes, nb_ctes_csv, fh_de_asignacion

('17', '02', '2026', '202602', 'report_20260217_c1.csv', '17-02-2026')

## Paqueterías

In [3]:
import os

from_drive = True  # same flag you use everywhere

if os.environ.get("HERMES_BOOTSTRAPPED") != "1":
    # ---------- GIT ON COLAB ONLY ----------
    try:
        from google.colab import userdata

        git_token = userdata.get('gitToken')
        git_user = userdata.get('gitUser')
        git_url = f'https://{git_token}@github.com/rene-aum/Hermes.git'
        branch_to_pull = 'dev'

        os.chdir('/content')

        if not os.path.isdir('Hermes'):
            !git clone {git_url}

        %cd Hermes
        !git fetch origin {branch_to_pull}
        !git checkout {branch_to_pull}
        !git pull origin {branch_to_pull}

        !pip install -r utils/src/requirements.txt
        %cd Asignacion

    except Exception as e:
        print(e)
        print('Running in other environment not colab probably!')

    # ---------- DRIVE + SHEETS ----------
    if from_drive:
        from pydrive2.auth import GoogleAuth
        from pydrive2.drive import GoogleDrive
        from google.colab import auth
        from oauth2client.client import GoogleCredentials
        import gspread
        from google.auth import default
        from gspread_dataframe import set_with_dataframe
        import gdown

        auth.authenticate_user()
        gauth = GoogleAuth()
        gauth.credentials = GoogleCredentials.get_application_default()
        drive = GoogleDrive(gauth)

        creds, _ = default()
        gc = gspread.authorize(creds)

    os.environ["HERMES_BOOTSTRAPPED"] = "1"
else:
    print("Bootstrap already done, assuming orchestrator ran it.")

/content/Hermes
From https://github.com/rene-aum/Hermes
 * branch            dev        -> FETCH_HEAD
Already on 'dev'
Your branch is up to date with 'origin/dev'.
From https://github.com/rene-aum/Hermes
 * branch            dev        -> FETCH_HEAD
Already up to date.
/content/Hermes/Asignacion


In [4]:
import sys
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import warnings
import sys
sys.path.append('..')
sys.path.append('../..')
from utils.utils import (get_dates_dataframe,
                       add_year_week,
                       custom_read,
                       process_columns,
                       remove_accents)

from utils.drive_toolbox import(from_drive_to_local,
                             get_last_modification_date_drive,
                             create_sheets_in_drive_folder,
                             update_sheets_in_drive_folder,
                             read_from_google_sheets,
                             list_file_ids_for_drive_folder,
                             create_csv_file_in_drive_folder,
                             write_csv_to_drive,
                             read_csv_from_drive,
                             append_dataframe_to_google_sheet_from_range)
from utils.src.constants import (atlas_consumo_output_folder_id,
                           consumo_sheets_ids_dict,
                           folder_id_bauto_gabo,
                           id_reporte_ventas,
                           id_edas_referenciados,
                           id_torre_de_control
                           )


warnings.filterwarnings('ignore')



In [5]:
# EXTRAS

# --------- BUSCAR EN SUBCARPETAS -------------------------------------------
from googleapiclient.discovery import build
creds, _ = default()

servicedrive = build("drive", "v3", credentials=creds)
service_sheets = build("sheets", "v4", credentials=creds)

FOLDER_MIME = "application/vnd.google-apps.folder"

def listar_archivos(folder_id, mime_types=None):
    """
    folder_id: ID de la carpeta raíz
    mime_types: None | string | lista de strings
    """
    if isinstance(mime_types, str):
        mime_types = [mime_types]

    resultados = {}

    def recorrer(fid):
        page_token = None
        while True:
            resp = servicedrive.files().list(
                q=f"'{fid}' in parents and trashed = false",
                fields="nextPageToken, files(id, name, mimeType)",
                pageToken=page_token,
                supportsAllDrives=True,
                includeItemsFromAllDrives=True
            ).execute()

            for f in resp.get("files", []):
                if f["mimeType"] == FOLDER_MIME:
                    recorrer(f["id"])
                else:
                    if mime_types is None or f["mimeType"] in mime_types:
                        resultados[f["name"]] = f["id"]

            page_token = resp.get("nextPageToken")
            if not page_token:
                break

    recorrer(folder_id)
    return resultados


# -------------- LEER CON ENCODING ------------------------------------------
import io
import pandas as pd
from googleapiclient.http import MediaIoBaseDownload

def read_csv_from_drive_v3(drive_service, file_id, **read_csv_kwargs):
    request = drive_service.files().get_media(fileId=file_id)
    fh = io.BytesIO()
    downloader = MediaIoBaseDownload(fh, request)

    done = False
    while not done:
        status, done = downloader.next_chunk()

    fh.seek(0)
    return pd.read_csv(fh, **read_csv_kwargs)


# import io
# def read_csv_from_drive2(drive, file_id, encoding="latin-1", **read_csv_kwargs):
#     f = drive.CreateFile({"id": file_id})
#     f.FetchContent()
#     b = f.content.getvalue()  # bytes
#     return pd.read_csv(io.BytesIO(b), encoding=encoding, **read_csv_kwargs)

# ------------- Todas las columnas ------------------------------------------
pd.set_option('display.max_columns', 100)

# ------------- IMPRIMIR CON COLORES ----------------------------------------
class color:
   PURPLE = '\033[95m'
   CYAN = '\033[94m'
   DARKCYAN = '\033[36m'
   BLUE = '\033[94m'
   GREEN = '\033[92m'
   YELLOW = '\033[93m'
   RED = '\033[91m'
   BOLD = '\033[1m'
   UNDERLINE = '\033[4m'
   END = '\033[0m'
import math
from zoneinfo import ZoneInfo

def borrar_hojas(spreadsheet_id, nb_hojas):
    spreadsheet = service_sheets.spreadsheets().get(
        spreadsheetId=spreadsheet_id
    ).execute()

    sheet_ids = []
    for sheet in spreadsheet["sheets"]:
        if sheet["properties"]["title"] in nb_hojas:
            sheet_ids.append(sheet["properties"]["sheetId"])

    request = {
        "requests": [
            {"deleteSheet": {"sheetId": s_id}}
            for s_id in sheet_ids
            ]
    }

    service_sheets.spreadsheets().batchUpdate(
        spreadsheetId=spreadsheet_id,
        body=request
    ).execute()
    print(f"{nb_hojas} eliminada(s)")
    return

    print("Hoja no encontrada")

def crear_hojas_sheets(spreadsheet_id, nb_hojas, quitar_cuadricula = True, fila_congelada = 1):
  """
  nb_hojas: lista con los nombres de las hojas a crear
  quitar_cuadricula: True | False es para hacer invisibles los bordes/cuadrícula de las celdas
  """

  request = {
      "requests": [
          {"addSheet": {"properties": {"title": nombre,
                                       'gridProperties': {'hideGridlines':quitar_cuadricula,
                                                          'frozenRowCount':fila_congelada}
                                       }
                        }
           }
          for nombre in nb_hojas
      ]
  }

  service_sheets.spreadsheets().batchUpdate(
      spreadsheetId = id_sheets_salida,
      body=request
  ).execute()
  print(f'{nb_hojas} creadas')


def formato_hojas_sheets(sheets_id, nb_hojas, n_columnas, tamanio_letra = 11, letra = 'Source Serif 4', rgb_encabezado = [0.1, 0.3, 0.7], ):
  spreadsheet = service_sheets.spreadsheets().get(
      spreadsheetId=sheets_id
  ).execute()

  sheet_ids = []
  for sheet in spreadsheet["sheets"]:
      if sheet["properties"]["title"] in nb_hojas:
          sheet_ids.append(sheet["properties"]["sheetId"])

  # requests de formato
  requests = []
  for sh_id in sheet_ids:
    # A) Fuente para TODA la hoja
    r_global = {
        "repeatCell": {
            "range": {
                "sheetId": sh_id
            },
            "cell": {
                "userEnteredFormat": {
                    "textFormat": {
                        "fontFamily": letra,
                        "fontSize": tamanio_letra
                    }
                }
            },
            "fields": "userEnteredFormat.textFormat(fontFamily,fontSize)"
        }
    },

    # B) Color solo en encabezado (fila 1)
    r_encabezado = {
        "repeatCell": {
            "range": {
                "sheetId": sh_id,
                "startRowIndex": 0,
                "endRowIndex": 1,
                'startColumnIndex':0,
                'endColumnIndex':n_columnas
            },
            "cell": {
                "userEnteredFormat": {
                    "backgroundColor": {
                        "red": rgb_encabezado[0],
                        "green": rgb_encabezado[1],
                        "blue": rgb_encabezado[2]
                    },
                    "textFormat": {
                        "bold": True,
                        "foregroundColor": {
                            "red": 1,
                            "green": 1,
                            "blue": 1
                        }
                    }
                }
            },
            "fields": "userEnteredFormat(backgroundColor,textFormat.bold,textFormat.foregroundColor)"
        }
    }
    r_anchoColumnas = {
        "autoResizeDimensions": {
            "dimensions": {
                "sheetId": sh_id,
                "dimension": "COLUMNS",
                "startIndex": 0,
                "endIndex": 20   # ajusta primeras 20 columnas
            }
        }
    }
    requests.append(r_global)
    requests.append(r_encabezado)
    requests.append(r_anchoColumnas)

  service_sheets.spreadsheets().batchUpdate(
      spreadsheetId=sheets_id,
      body={"requests": requests}
      ).execute()

## Leemos base de solicitudes y nos quedamos con los nuevos desde el último corte

In [6]:
nb_arch_sols = 'Solicitudes de crédito 01-08-2025 al '
dict_SolicitudesCredito_id = { x: v for x, v in
                              listar_archivos(id_drive_apisFolios,
                                              mime_types = 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet').items()
                                              if nb_arch_sols in x}
dict_SolicitudesCredito_id = {
    x.split(nb_arch_sols)[1][:13]: v
    for x, v in dict_SolicitudesCredito_id.items() if 'Revisión' not in x
}

fto_fh = '%d-%m-%Y-%H'
fhs_sols = list(dict_SolicitudesCredito_id.keys())
fhs_sols = sorted([datetime.strptime(x, fto_fh) for x in fhs_sols], reverse=True)

ults_sols = fhs_sols[0].strftime(fto_fh)
pens_sols = fhs_sols[1].strftime(fto_fh)
print(f'Vamos a cargar solicitudes de crédito entre {color.DARKCYAN} {fhs_sols[0]} y {fhs_sols[1]} {color.END} (ignorando minutos)')

Vamos a cargar solicitudes de crédito entre  2026-02-17 09:00:00 y 2026-02-16 09:00:00  (ignorando minutos)


In [7]:
from_drive_to_local(drive, dict_SolicitudesCredito_id[ults_sols], ults_sols)
from_drive_to_local(drive, dict_SolicitudesCredito_id[pens_sols], pens_sols)

sols_sf_ult = pd.read_excel(ults_sols)
sols_sf_penult = pd.read_excel(pens_sols)

sols_sf_ult.rename(columns = {'Name':'id_sol','MX_ATN_Account__r.Name':'nb_comprador','MX_ATN_Account__r.MX_ATN_CommerceId__c':'id_am','MX_ATN_Account__r.MX_ATN_PrimaryContact__r.Email':'email'
          , 'MX_ATN_Account__r.MX_ATN_PrimaryContact__r.MobilePhone':'fon','MX_ATN_Status__c':'status_solicitud','CreatedDate':'fh_creacion','MX_ATN_creditId__c':'folio'},inplace=True)
sols_sf_ult = sols_sf_ult[~sols_sf_ult['status_solicitud'].str.contains('Rechazada')]
sols_sf_ult['fh_creacion'] = pd.to_datetime(sols_sf_ult['fh_creacion'].str[:10], format='%d/%m/%Y')
sols_sf_ult = (sols_sf_ult[pd.to_datetime(sols_sf_ult['fh_creacion']) >= datetime(2025,7,1)]
               ).sort_values(by='id_sol').reset_index(drop=True)

sols_sf_penult.rename(columns = {'Name':'id_sol','MX_ATN_Account__r.Name':'nb_comprador','MX_ATN_Account__r.MX_ATN_CommerceId__c':'id_am','MX_ATN_Account__r.MX_ATN_PrimaryContact__r.Email':'email'
          , 'MX_ATN_Account__r.MX_ATN_PrimaryContact__r.MobilePhone':'fon','MX_ATN_Status__c':'status_solicitud','CreatedDate':'fh_creacion','MX_ATN_creditId__c':'folio'},inplace=True)
sols_sf_penult = sols_sf_penult[~sols_sf_penult['status_solicitud'].str.contains('Rechazada')]
sols_sf_penult['fh_creacion'] = pd.to_datetime(sols_sf_penult['fh_creacion'].str[:10], format='%d/%m/%Y')
sols_sf_penult = (sols_sf_penult[pd.to_datetime(sols_sf_penult['fh_creacion']) >= datetime(2025,7,1)]
               ).sort_values(by='id_sol').reset_index(drop=True)

sols_sf = sols_sf_ult[~sols_sf_ult['id_sol'].isin(sols_sf_penult['id_sol'])].copy().reset_index(drop=True) # Hacemos la diferencia vs el día previo
sols_sf['tp_solicitud'] = np.where(sols_sf['folio'].isna(),'Contingencia Crédito','Apificado Crédito')

display(sols_sf.sample())
print(f'{color.BOLD}{color.CYAN}Tenemos {sols_sf.shape[0]} solicitudes nuevas{color.END}')
print(f'{color.BOLD}{color.GREEN}{len(sols_sf[sols_sf['tp_solicitud']=='Apificado Crédito'])} de API y {color.BLUE}{len(sols_sf[sols_sf['tp_solicitud']=='Contingencia Crédito'])} de Contingencia')

,id_sol,MX_ATN_Id_Simulacion__c,folio,fh_creacion,MX_ATN_SolicitudBAuto__c,nb_comprador,id_am,email,fon,status_solicitud,LastModifiedDate,tp_solicitud
10,SC-13849,50103304,NaN,2026-02-16,NaN,BRIAN JESUS,595096,brianmurillo281@gmail.com,5.256445e+11,En Proceso,"16/02/2026, 14:54",Contingencia Crédito


Tenemos 34 solicitudes nuevas
11 de API y 23 de Contingencia


In [8]:
# CELDA DE VALIDACIÓN
fh_final_sols, fh_inicial_sols = fhs_sols[0].replace(hour=0), fhs_sols[1].replace(hour=0)
fhs_creacion_ls = sols_sf['fh_creacion'].unique().tolist()

if not all([x <= fh_final_sols and x >= fh_inicial_sols for x in fhs_creacion_ls]):
    print(f"{color.RED}Hay solicitudes con fechas fuera del rango de actualización{color.END}")
    raise SystemExit

## Pegamos datos de clientes

In [9]:
pdds_sf = sols_sf.copy()

In [10]:
id_drive_ctes_mes = list_file_ids_for_drive_folder(drive, id_drive_ctes)[f'{nb_carpeta_ctes_mes}']
archs_ctesnvos = list_file_ids_for_drive_folder(drive, id_drive_ctes_mes)

ctes_nvos = read_csv_from_drive_v3(servicedrive, archs_ctesnvos[nb_ctes_csv], encoding = 'latin-1')
print(f'{color.BOLD}{color.CYAN}Se cargó base {nb_ctes_csv}{color.END}')
# ctes_nvos['fh_actualizacion'] = arch_reciente[0]
ctes_nvos.rename(columns = {'Id Comercio Externo':'id_am', 'Teléfono':'phone', 'Email':'email'}, inplace=True)

ctes_nvos = ctes_nvos[~ctes_nvos['id_am'].isna()]
ctes_nvos['id_am'] = ctes_nvos['id_am'].astype(int)
ctes_nvos = ctes_nvos[['id_am','phone','email']]
ctes_nvos['origen_arch'] = 'ctes_sf'
ctes_nvos.sample()

Se cargó base report_20260217_c1.csv


,id_am,phone,email,origen_arch
70037,391176,525545153883.0,francisco.lopez.camacho.contractor@bbva.com,ctes_sf


In [11]:
# Aqui debo dar prioridad a la base AcClientes sobre la de andrés, porque hay registros que ahí están más completos (comentario de Rene)
ctes = read_from_google_sheets(gc, consumo_sheets_ids_dict['AcClientes'])
ctes['origen_arch'] = 'ctes_ac'

ctes = pd.concat([ctes, ctes_nvos])
ctes = ctes.drop_duplicates('id_am',keep='first').reset_index(drop=True)
ctes['phone'] = pd.to_numeric(ctes['phone'].fillna(0),errors='coerce').astype('Int64').astype(str).apply(lambda x: x[-10:])
ctes['id_am'] = ctes['id_am'].astype(int)

display(ctes.sample())
print(f'{color.BOLD}{color.CYAN}Tenemos {ctes.shape[0]} clientes{color.END}')

,id_am,billing_firstname,billing_lastname,nickname,email,phone,zip,country,state_province,customer_since,date_of_birth,phone_number_otp_validated,email_otp_validated,origen_arch
116552,44973,DIEGO DUR√°N ROJAS,DIEGO DUR√°N ROJAS,DIEGO DUR√°N ROJAS,ddrojas29@gmail.com,7228857012,6600.0,Mexico,Ciudad de M√©xico,2025-01-12,NaN,1.0,1.0,ctes_ac


Tenemos 199553 clientes


In [12]:
# Pegamos info de clientes a los nuevos pedidos
pdds_sf['id_am'] = pdds_sf['id_am'].astype(int)
pdds_sf1 = pdds_sf.merge(ctes[['id_am','email','phone','origen_arch']], how = 'left', on = 'id_am',suffixes = ['_borigen',''])
pdds_sf1.sample(1)

,id_sol,MX_ATN_Id_Simulacion__c,folio,fh_creacion,MX_ATN_SolicitudBAuto__c,nb_comprador,id_am,email_borigen,fon,status_solicitud,LastModifiedDate,tp_solicitud,email,phone,origen_arch
18,SC-13859,50111311,NaN,2026-02-16,NaN,Lucia jimenez,600268,wailani1191@gmail.com,5.255320e+11,En Proceso,"16/02/2026, 18:15",Contingencia Crédito,wailani1191@gmail.com,5532047174,ctes_ac


In [13]:
if not pdds_sf1[(pdds_sf1['email'].isna()) | (pdds_sf1['phone'].isna())].shape[0] == 0:
    print(f"{color.RED}Hay pedidos sin datos de comprador{color.END}")
    raise SystemExit

## Buscamos datos de solicitudes nuevas en histórico y nos quedamos con las que cumplan definición de leads nuevos

In [34]:
# Leemos y ordenamos histórico, corregimos nombre de asesora y generamos variable de último lead
id_hist = '1zvW-Dxow9gz1Dnbpg_jO7my4wadDvJDW'
nb_hist = '_latest.csv'
hist = list_file_ids_for_drive_folder(drive, id_hist)
hist = [v for i,v in hist.items() if nb_hist in i ][0]
hist = read_csv_from_drive_v3(servicedrive, hist, encoding='latin-1' )

# Primero los registros más recientes
hist['fecha de asignacion'] = pd.to_datetime(hist['fecha de asignacion'], format = '%Y-%m-%d')
hist = hist.sort_values(by='fecha de asignacion', ascending=False)
#------------------------------------

hist['asesor espacio'] = hist['asesor espacio'].str.replace('SaldaÃ±a','Saldaña')
hist['conteo_leads'] = hist['id lead'].str.replace('|','0').str[-6:].astype(int)
ultimo_lead = hist['conteo_leads'].max()
print(f'{color.BLUE}El último lead, a partir del cual vamos a empezar a asignar en este proceso es el {ultimo_lead}{color.END}')
hist.sample(1)

El último lead, a partir del cual vamos a empezar a asignar en este proceso es el 17974


,id lead,origen automarket,cosecha,id comprador,folio bauto tc,nombre comprador,mail comprador,telefono comprador,asesor credito,espacio automarket,asesor espacio,fecha de asignacion,estatus de lead,fecha_de_proceso,flag_torre_v2,flag salio de cerrado,fecha de reactivacion credito,fecha de reactivacion eam,conteo_leads
15154,LAA-014450,Apartado,Cosecha Ene 26,546913,x,MARCO ANTONIO PEREZ PINEDA,pinedaperezmarco24@gmail.com,5562083569,Cintya Aguirre,MetrÃ³poli Patriotismo,Georgina Zeferino,2026-01-22,CERRADO,2026-02-05 10:07:01,1.0,NaN,NaN,NaN,14450


In [15]:
hist_id = hist[['id comprador','id lead','estatus de lead']].copy().drop_duplicates('id comprador').dropna(subset='id comprador')
hist_mail = hist[['mail comprador','id lead','estatus de lead']].copy().drop_duplicates('mail comprador').dropna(subset='mail comprador')
hist_fon = hist[['telefono comprador','id lead','estatus de lead']].copy().drop_duplicates('telefono comprador').dropna(subset='telefono comprador')

hist_id.columns = ['id_comprador','id_lead','estatus_lead']
hist_mail.columns = ['email','id_lead','estatus_lead']
hist_fon.columns = ['phone','id_lead','estatus_lead']

pdds_sf1['id_comprador'] = pdds_sf1['id_am'].astype(str)
pdds_sf1['phone'] = pd.to_numeric(pdds_sf1['phone'].astype(str).str[-10:],errors = 'coerce').astype('Int64')
hist_fon['phone'] = pd.to_numeric(hist_fon['phone'].astype(str).str[-10:],errors='coerce').astype('Int64')
pdds_sf2 = pdds_sf1.merge(hist_id[['id_comprador','id_lead','estatus_lead']], how = 'left', on = 'id_comprador', suffixes = ['','_conid'])
pdds_sf2 = pdds_sf2.merge(hist_mail[['email','id_lead','estatus_lead']], how = 'left', on = 'email', suffixes = ['','_conemail'])
pdds_sf2 = pdds_sf2.merge(hist_fon[['phone','id_lead','estatus_lead']], how = 'left', on = 'phone', suffixes = ['','_confon'])

pdds_sf2['id_lead'] = np.where(pdds_sf2['id_lead'].notna(), pdds_sf2['id_lead'],
                               np.where(pdds_sf2['id_lead_conemail'].notna(), pdds_sf2['id_lead_conemail'],
                                        np.where(pdds_sf2['id_lead_confon'].notna(), pdds_sf2['id_lead_confon'],np.nan)))

total_congruentes_incongruentes = len(pdds_sf2)
cerrados = ['COMPRA EXITOSA ','COMPRA EXITOSA','CERRADO','NA']
pdds_sf2['aux'] =((~pdds_sf2['estatus_lead'].fillna('NA').isin(cerrados) )*1 + (~pdds_sf2['estatus_lead_conemail'].fillna('NA').isin(cerrados))*1 +
                  (~pdds_sf2['estatus_lead_confon'].fillna('NA').isin(cerrados))*1)
pdds_stts_incongruente = pdds_sf2[pdds_sf2['aux'].between(1,2)].copy()
pdds_sf2 = pdds_sf2[pdds_sf2['aux'].isin([0,3])].copy()

pdds_sf2['estatus_lead'] = np.where(pdds_sf2['estatus_lead'].notna(), pdds_sf2['estatus_lead'],
                               np.where(pdds_sf2['estatus_lead_conemail'].notna(), pdds_sf2['estatus_lead_conemail'],
                                        np.where(pdds_sf2['estatus_lead_confon'].notna(), pdds_sf2['estatus_lead_confon'],np.nan)))

nvos_leads = ( (pdds_sf2['id_lead'].isna()) | (pdds_sf2['estatus_lead'].isin(cerrados) ) )

leads_ok = pdds_sf2[~nvos_leads].copy()
leads_ok = pd.concat([leads_ok, pdds_stts_incongruente])

# leads_nvos = pdds_sf2.copy()
# leads_nvos = leads_nvos[~leads_nvos['num_pedido'].isin(pedidos_multicontacto['num_pedido'].unique())]
leads_nvos = pdds_sf2[nvos_leads].copy()

print(len(leads_ok), len(leads_nvos), total_congruentes_incongruentes)
display(pdds_stts_incongruente)

11 23 34


,id_sol,MX_ATN_Id_Simulacion__c,folio,fh_creacion,MX_ATN_SolicitudBAuto__c,nb_comprador,id_am,email_borigen,fon,status_solicitud,LastModifiedDate,tp_solicitud,email,phone,origen_arch,id_comprador,id_lead,estatus_lead,id_lead_conemail,estatus_lead_conemail,id_lead_confon,estatus_lead_confon,aux


In [16]:
if not len(leads_ok) + len(leads_nvos) == total_congruentes_incongruentes:
    print(f"{color.RED}El total de pedidos que requieren leads, considerando los incongruentes, no cuadra con el total de pedidos iniciales{color.END}")
    print(f"{color.BOLD}{color.CYAN}La clasificación de leads ok (pedidos que no requieren lead nuevo) y leads nuevos está perdiendo alguno(s) de los pedidos con los que iniciamos {color.END}")
    raise SystemExit

In [17]:
leads_nvos = leads_nvos[['id_comprador','phone','email','nb_comprador','tp_solicitud','folio']].drop_duplicates(['id_comprador','tp_solicitud'], keep='last').reset_index(drop=True)

rep_multiorigen = leads_nvos.copy()[leads_nvos.duplicated('id_comprador', keep=False)]
print(f'{color.BOLD}{color.DARKCYAN} Se encontraron {len(rep_multiorigen.id_comprador.unique())} leads con multiorigen (apis y contingencia): {color.END}')
display(rep_multiorigen)

# Priorizamos etiqueta de apificados
priorizacion_origen = ['Apificado Crédito',' Contingencia Crédito']
leads_nvos = leads_nvos.sort_values('tp_solicitud',
                                    key = lambda x: pd.Categorical(x, categories = priorizacion_origen, ordered = True))
leads_nvos = leads_nvos.drop_duplicates('id_comprador', keep='first').reset_index(drop=True)
print(f'{color.BOLD}{color.DARKCYAN} Priorizamos etiqueta de APIs. {color.END}')
print(f'{color.BOLD}{color.CYAN} Tenemos {len(leads_nvos)} leads nuevos.{color.END}')

 Se encontraron 0 leads con multiorigen (apis y contingencia): 


,id_comprador,phone,email,nb_comprador,tp_solicitud,folio


 Priorizamos etiqueta de APIs. 
 Tenemos 21 leads nuevos.


## Asignaciones

### Asignación espacio

In [18]:
# Primero asignamos espacio

hist_credito_activos = hist.copy()
hist_credito_activos = hist_credito_activos[hist_credito_activos['origen automarket'].str.contains(r'Contingencia|Apificado|API') &
                                             ((~hist_credito_activos['estatus de lead'].isin(cerrados)) & (hist_credito_activos['espacio automarket']!='PRUEBA'))]

leads_credito_activos = hist_credito_activos.groupby(['espacio automarket']).agg({'id lead':'nunique'}).reset_index()
leads_credito_activos = leads_credito_activos.sort_values(by='id lead').reset_index(drop=True)
leads_credito_activos['diff_leads'] = leads_credito_activos['id lead'].diff().shift(-1).fillna(0).astype(int)
leads_credito_activos

,espacio automarket,id lead,diff_leads
0,Samara SatÃ©lite,88,6
1,Reforma 510,94,1
2,MetrÃ³poli Patriotismo,95,0


In [19]:
# df de asignación para emparejar los leads activos en espacios
asign_espacio_justiciera = leads_credito_activos.loc[
    leads_credito_activos.index.repeat(leads_credito_activos["diff_leads"])
]['espacio automarket'].reset_index(drop=True) # sale como una serie
asign_espacio_justiciera = asign_espacio_justiciera.to_frame()

# df de asignación normal
asign_espacio_normal = leads_credito_activos.loc[
    leads_credito_activos.index
]['espacio automarket'].reset_index(drop=True).to_frame()

# Ahora vamos a pegar tantas veces como sea necesario la asignación normal
lte = len(leads_nvos) - len(asign_espacio_justiciera) # leads tras emparejamiento
repeticiones_carrusel = ( math.ceil(lte/len(asign_espacio_normal)) ) if lte>0 else 0
print(f'Después de emparejar los leads de crédito en cada espacio, vamos a pasarlos {repeticiones_carrusel} veces por el carrusel')

asign_espacio = asign_espacio_justiciera.copy()
for i in range(repeticiones_carrusel):
  asign_espacio = pd.concat([asign_espacio,asign_espacio_normal])
asign_espacio.reset_index(drop=True, inplace = True)
asign_espacio['llave_espacio'] = asign_espacio.index + 1
asign_espacio

Después de emparejar los leads de crédito en cada espacio, vamos a pasarlos 5 veces por el carrusel


,espacio automarket,llave_espacio
0,Samara SatÃ©lite,1
1,Samara SatÃ©lite,2
2,Samara SatÃ©lite,3
3,Samara SatÃ©lite,4
4,Samara SatÃ©lite,5
5,Samara SatÃ©lite,6
6,Reforma 510,7
7,Samara SatÃ©lite,8
8,Reforma 510,9
9,MetrÃ³poli Patriotismo,10


In [20]:
leads_nvos = leads_nvos.reset_index(drop=True)
leads_nvos['llave_espacio'] = leads_nvos.index + 1
leads_nvos = leads_nvos.merge(asign_espacio, how='left', on = 'llave_espacio')
leads_nvos['espacio automarket'] = leads_nvos['espacio automarket'].replace(dicc_espacios)

### Asignacion asesores

In [21]:
# Leemos catálogo de asesores
cat_as = list_file_ids_for_drive_folder(drive, id_drive_catalogos)['AsesoresEspacio']
centros = ['torre','samara','patriotismo','celula_credito']
assrs_actvs = {}
for c in centros:
  df = read_from_google_sheets(gc,cat_as,c)
  df['espacio'] = c
  df = df[df.activo==1].drop_duplicates(['asesor','espacio'])
  assrs_actvs[c] = df
assrs_actvs = pd.concat(assrs_actvs.values())
assrs_actvs = assrs_actvs.reset_index(drop=True)
assrs_actvs['asesor'] = assrs_actvs['asesor'].str.lower().str.strip()

# Número de asesores activos por espacio
num_assrs = {c: len(assrs_actvs[assrs_actvs.espacio==c]) for c in assrs_actvs.espacio.unique()}
num_assrs

{'torre': 5, 'samara': 4, 'patriotismo': 5, 'celula_credito': 4}

In [22]:
# Leads activos por asesor

leads_asesor_espacio = hist[~hist['estatus de lead'].isin(['COMPRA EXITOSA ','COMPRA EXITOSA', 'CERRADO'])].groupby(['asesor espacio','espacio automarket']).agg({'id lead':'nunique'}).reset_index()
leads_asesor_espacio.columns = ['asesor','espacio','leads']
leads_asesor_espacio['asesor'] = leads_asesor_espacio['asesor'].str.lower().str.strip()
leads_asesor_espacio['espacio'] = leads_asesor_espacio['espacio'].replace(dicc_espacios)
leads_asesor_espacio = leads_asesor_espacio[~leads_asesor_espacio['asesor'].str.strip().isin(['PRUEBA','prueba','#N/A()','#n/a ()'])]
leads_asesor_espacio = leads_asesor_espacio.groupby(['asesor','espacio']).agg({'leads':'sum'}).reset_index()

leads_asesor_cred = hist[~hist['estatus de lead'].isin(['COMPRA EXITOSA ','COMPRA EXITOSA', 'CERRADO'])].groupby(['asesor credito']).agg({'id lead':'nunique'}).reset_index()
leads_asesor_cred.columns = ['asesor','leads']
leads_asesor_cred['asesor'] = leads_asesor_cred['asesor'].str.lower().str.strip()
leads_asesor_cred['espacio'] = 'celula_credito'
leads_asesor_cred = leads_asesor_cred[~leads_asesor_cred['asesor'].str.strip().isin(['PRUEBA', 'prueba', '#N/A()', '#n/a ()', '', ' '])]
leads_asesor_cred = leads_asesor_cred.groupby(['asesor','espacio']).agg({'leads':'sum'}).reset_index()

leads_asesor = pd.concat([leads_asesor_espacio, leads_asesor_cred])

assrs_actvs_leads = assrs_actvs.merge(leads_asesor, how = 'left', on = ['asesor','espacio'])
assrs_actvs_leads = assrs_actvs_leads.sort_values(['espacio','leads']).reset_index(drop=True)

assrs_actvs_leads['llave'] = assrs_actvs_leads.groupby('espacio').cumcount()
assrs_actvs_leads['leads'] = assrs_actvs_leads['leads'].fillna(0)

leads_asesor_cred = assrs_actvs_leads[assrs_actvs_leads['espacio'] == 'celula_credito'].rename(columns = {'asesor':'asesor credito'})
leads_asesor_esp = assrs_actvs_leads[assrs_actvs_leads['espacio'] != 'celula_credito'].rename(columns = {'asesor':'asesor espacio'})
display(leads_asesor_cred)
display(leads_asesor_esp)

,asesor credito,activo,espacio,leads,llave
0,cintya aguirre,1.0,celula_credito,409,0
1,adrian gutierrez,1.0,celula_credito,423,1
2,karla herrera,1.0,celula_credito,453,2
3,jose luis acevedo,1.0,celula_credito,469,3


,asesor espacio,activo,espacio,leads,llave
4,lilian juarez,1.0,patriotismo,100,0
5,georgina zeferino,1.0,patriotismo,125,1
6,patricia aldana,1.0,patriotismo,129,2
7,daniel garcia,1.0,patriotismo,143,3
8,adonahi rueda,1.0,patriotismo,151,4
9,erick hernandez,1.0,samara,133,0
10,guadalupe diaz,1.0,samara,136,1
11,monica rodriguez,1.0,samara,137,2
12,cynthia pacheco,1.0,samara,139,3
13,ximena gutierrez,1.0,torre,107,0


In [23]:
leads_nvos = leads_nvos.rename(columns = {'espacio automarket':'espacio asig'}) #espacio asig es el espacio que asignamos en el paso previo
# Vamos a procesar distinto los leads que ya tuvieron gestión en algún espacio. Los seleccionamos con inner merge vs histórico

leads_nvos_comprprevio = leads_nvos.copy()
leads_nvos_comprprevio = leads_nvos_comprprevio.merge(
    hist[['id comprador','espacio automarket','asesor espacio']].rename(
    columns = {'espacio automarket':'espacio previo', 'asesor espacio':'asesor espacio previo'}
    ).drop_duplicates(
        'id comprador',keep='first'),
                                        how = 'inner', left_on = 'id_comprador', right_on = 'id comprador').drop(columns = ['id comprador'])

leads_nvos_comprprevio['espacio previo'] = leads_nvos_comprprevio['espacio previo'].replace(dicc_espacios)

# Damos por bueno el espacio que tenía en su lead anterior; mergeamos por espacio previo con catálogo de asesores y nos quedamos con el primer asesor que cruce y no haya tenido antes (columna aux)
leads_nvos_comprprevio = leads_nvos_comprprevio.merge(leads_asesor_esp.rename(columns = {'asesor espacio':'asesor espacio nvo'}), how = 'left', left_on = 'espacio previo', right_on = 'espacio')
leads_nvos_comprprevio['aux'] = (leads_nvos_comprprevio['asesor espacio previo'].str.lower().str.strip() == leads_nvos_comprprevio['asesor espacio nvo'].str.lower().str.strip())*1
leads_nvos_comprprevio = leads_nvos_comprprevio.sort_values(by='aux', ascending=True)
leads_nvos_comprprevio = leads_nvos_comprprevio.drop_duplicates(['id_comprador'], keep = 'first').reset_index(drop=True)
leads_nvos_comprprevio = leads_nvos_comprprevio.drop(columns = ['espacio asig','espacio previo','asesor espacio previo','activo','leads','llave','aux']).rename(columns = {'asesor espacio nvo':'asesor espacio'})
display(leads_nvos_comprprevio)

,id_comprador,phone,email,nb_comprador,tp_solicitud,folio,llave_espacio,asesor espacio,espacio
0,447859,3315114743,rsambo198@gmail.com,Raymond Sambo,Contingencia Crédito,NaN,19,lilian juarez,patriotismo


In [24]:
# Aqui procesamos los leads nuevos con comprador nuevo
leads_nvos_comprnvo = leads_nvos[~leads_nvos['id_comprador'].isin(leads_nvos_comprprevio['id_comprador'].unique())].copy()
leads_nvos_comprnvo['llave_esp'] = leads_nvos_comprnvo.groupby('espacio asig').cumcount() % leads_nvos_comprnvo['espacio asig'].map(num_assrs)
leads_nvos_comprnvo = leads_nvos_comprnvo.merge(leads_asesor_esp, how='left', left_on = ['espacio asig','llave_esp'], right_on = ['espacio','llave'])
leads_nvos_comprnvo

,id_comprador,phone,email,nb_comprador,tp_solicitud,folio,llave_espacio,espacio asig,llave_esp,asesor espacio,activo,espacio,leads,llave
0,273871,5520237718,jan.mar.usoi@gmail.com,JANETTE MONTSERRAT,Apificado Crédito,9624668,1,samara,0,erick hernandez,1.0,samara,133,0
1,599638,5533357288,cgv24@msn.com,CARLOS ENRIQUE,Apificado Crédito,9624722,2,samara,1,guadalupe diaz,1.0,samara,136,1
2,150414,5530204356,chambapaco2025@gmail.com,FRANCISCO,Apificado Crédito,9627135,3,samara,2,monica rodriguez,1.0,samara,137,2
3,572128,5566771159,vigtere901110@gmail.com,TERESA,Apificado Crédito,9628306,4,samara,3,cynthia pacheco,1.0,samara,139,3
4,600391,5522674422,yair4496@gmail.com,CESAR YAIR,Apificado Crédito,9628592,5,samara,0,erick hernandez,1.0,samara,133,0
5,580399,4445991077,ponce3358@gmail.com,JOSE RAYMUNDO,Contingencia Crédito,NaN,6,samara,1,guadalupe diaz,1.0,samara,136,1
6,599509,5649145819,cristian229512@gmail.com,CRISTIAN,Contingencia Crédito,NaN,7,torre,0,ximena gutierrez,1.0,torre,107,0
7,599479,5580323631,vmquijan@gmail.com,VICTOR MANUEL,Contingencia Crédito,NaN,8,samara,2,monica rodriguez,1.0,samara,137,2
8,599626,5636609585,sugardojuarez@gmail.com,Jorge Jonathan Juárez González,Contingencia Crédito,NaN,9,torre,1,ana karen castro,1.0,torre,108,1
9,595096,5644492113,brianmurillo281@gmail.com,BRIAN JESUS,Contingencia Crédito,NaN,10,patriotismo,0,lilian juarez,1.0,patriotismo,100,0


In [25]:
# Juntamos leads de compradores previos y nuevos, y asignamos asesor de credito
salida_leads = pd.concat([leads_nvos_comprnvo, leads_nvos_comprprevio]).sort_values(by='id_comprador').reset_index(drop=True)
salida_leads['llave_celcred'] = salida_leads.index % num_assrs['celula_credito']
salida_leads = salida_leads.merge(leads_asesor_cred, how = 'left', left_on = 'llave_celcred', right_on = 'llave', suffixes = ['','_cred'])
salida_leads

,id_comprador,phone,email,nb_comprador,tp_solicitud,folio,llave_espacio,espacio asig,llave_esp,asesor espacio,activo,espacio,leads,llave,llave_celcred,asesor credito,activo_cred,espacio_cred,leads_cred,llave_cred
0,150414,5530204356,chambapaco2025@gmail.com,FRANCISCO,Apificado Crédito,9627135,3,samara,2.0,monica rodriguez,1.0,samara,137.0,2.0,0,cintya aguirre,1.0,celula_credito,409,0
1,273871,5520237718,jan.mar.usoi@gmail.com,JANETTE MONTSERRAT,Apificado Crédito,9624668,1,samara,0.0,erick hernandez,1.0,samara,133.0,0.0,1,adrian gutierrez,1.0,celula_credito,423,1
2,328938,5532310600,valebestgo@hotmail.com,Martha Angelica Martinez Torres,Contingencia Crédito,NaN,13,patriotismo,1.0,georgina zeferino,1.0,patriotismo,125.0,1.0,2,karla herrera,1.0,celula_credito,453,2
3,447859,3315114743,rsambo198@gmail.com,Raymond Sambo,Contingencia Crédito,NaN,19,NaN,NaN,lilian juarez,NaN,patriotismo,NaN,NaN,3,jose luis acevedo,1.0,celula_credito,469,3
4,538339,5610865851,dostinplay@gmail.com,DOSTIN JERAMINE,Contingencia Crédito,NaN,17,samara,1.0,guadalupe diaz,1.0,samara,136.0,1.0,0,cintya aguirre,1.0,celula_credito,409,0
5,557944,5543152303,jrservicio01@gmail.com,JESUS,Contingencia Crédito,NaN,18,torre,4.0,mariana saldaña,1.0,torre,143.0,4.0,1,adrian gutierrez,1.0,celula_credito,423,1
6,558850,5532799913,nietolizbeth@gmail.com,IVOCN LIZBETH,Contingencia Crédito,NaN,12,torre,2.0,yesenia cruz,1.0,torre,114.0,2.0,2,karla herrera,1.0,celula_credito,453,2
7,572128,5566771159,vigtere901110@gmail.com,TERESA,Apificado Crédito,9628306,4,samara,3.0,cynthia pacheco,1.0,samara,139.0,3.0,3,jose luis acevedo,1.0,celula_credito,469,3
8,580399,4445991077,ponce3358@gmail.com,JOSE RAYMUNDO,Contingencia Crédito,NaN,6,samara,1.0,guadalupe diaz,1.0,samara,136.0,1.0,0,cintya aguirre,1.0,celula_credito,409,0
9,595096,5644492113,brianmurillo281@gmail.com,BRIAN JESUS,Contingencia Crédito,NaN,10,patriotismo,0.0,lilian juarez,1.0,patriotismo,100.0,0.0,1,adrian gutierrez,1.0,celula_credito,423,1


## Formato de salidas

In [26]:
# Leemos catálogo de nomenclatura de leads
cat_nomLeads = list_file_ids_for_drive_folder(drive, id_drive_catalogos)['NomenclaturaLeads']
cat_nomLeads = read_from_google_sheets(gc, cat_nomLeads)
cat_nomLeads = cat_nomLeads[['Tipo de Lead','Clave']]

In [27]:
salida_leads = salida_leads.reset_index(drop=True)
salida_leads['index'] = salida_leads.index + 1
salida_leads['id lead'] = ultimo_lead + salida_leads['index']
salida_leads = salida_leads.merge(cat_nomLeads, how = 'left', left_on = 'tp_solicitud', right_on = 'Tipo de Lead')
# Validamos que todos tengan nomenclatura asignada
if (salida_leads['Clave'].isna().sum() > 0):
  print(f'{color.RED} Algo falló en la nomenclatura de leads. Hay algunos sin clave{color.END}')
  raise SystemExit

salida_leads['id lead'] = salida_leads['Clave'] + '-' + salida_leads['id lead'].astype(str).str.zfill(6)
salida_leads['folio'] = salida_leads['folio'].fillna('x')
salida_leads.drop(columns = ['llave','index','activo','leads'], inplace = True)
salida_leads['cosecha'] = cosecha
salida_leads['fecha de asignacion'] = fh_de_asignacion.replace('-','/')

salida_leads.rename(columns = {'id_comprador':'id comprador','espacio':'espacio automarket','phone':'telefono comprador','email':'mail comprador',
                               'nb_comprador':'nombre comprador','tp_solicitud':'origen automarket','folio':'folio bauto tc'},inplace=True)
salida_leads = salida_leads[['id lead','origen automarket','cosecha','id comprador','folio bauto tc',
                             'nombre comprador','mail comprador','telefono comprador','asesor credito','espacio automarket','asesor espacio','fecha de asignacion']]

salida_leads['espacio automarket'] = salida_leads['espacio automarket'].replace(dicc_espacios3)
salida_leads['asesor credito'] = salida_leads['asesor credito'].str.title()
salida_leads['asesor espacio'] = salida_leads['asesor espacio'].str.title()
salida_leads['estatus de lead'] = 'celula de credito'


salida_leads

,id lead,origen automarket,cosecha,id comprador,folio bauto tc,nombre comprador,mail comprador,telefono comprador,asesor credito,espacio automarket,asesor espacio,fecha de asignacion,estatus de lead
0,LAC-017897,Apificado Crédito,Cosecha Mar 26,150414,9627135,FRANCISCO,chambapaco2025@gmail.com,5530204356,Cintya Aguirre,Samara Satélite,Monica Rodriguez,17/02/2026,celula de credito
1,LAC-017898,Apificado Crédito,Cosecha Mar 26,273871,9624668,JANETTE MONTSERRAT,jan.mar.usoi@gmail.com,5520237718,Adrian Gutierrez,Samara Satélite,Erick Hernandez,17/02/2026,celula de credito
2,LCC-017899,Contingencia Crédito,Cosecha Mar 26,328938,x,Martha Angelica Martinez Torres,valebestgo@hotmail.com,5532310600,Karla Herrera,Metrópoli Patriotismo,Georgina Zeferino,17/02/2026,celula de credito
3,LCC-017900,Contingencia Crédito,Cosecha Mar 26,447859,x,Raymond Sambo,rsambo198@gmail.com,3315114743,Jose Luis Acevedo,Metrópoli Patriotismo,Lilian Juarez,17/02/2026,celula de credito
4,LCC-017901,Contingencia Crédito,Cosecha Mar 26,538339,x,DOSTIN JERAMINE,dostinplay@gmail.com,5610865851,Cintya Aguirre,Samara Satélite,Guadalupe Diaz,17/02/2026,celula de credito
5,LCC-017902,Contingencia Crédito,Cosecha Mar 26,557944,x,JESUS,jrservicio01@gmail.com,5543152303,Adrian Gutierrez,Reforma 510,Mariana Saldaña,17/02/2026,celula de credito
6,LCC-017903,Contingencia Crédito,Cosecha Mar 26,558850,x,IVOCN LIZBETH,nietolizbeth@gmail.com,5532799913,Karla Herrera,Reforma 510,Yesenia Cruz,17/02/2026,celula de credito
7,LAC-017904,Apificado Crédito,Cosecha Mar 26,572128,9628306,TERESA,vigtere901110@gmail.com,5566771159,Jose Luis Acevedo,Samara Satélite,Cynthia Pacheco,17/02/2026,celula de credito
8,LCC-017905,Contingencia Crédito,Cosecha Mar 26,580399,x,JOSE RAYMUNDO,ponce3358@gmail.com,4445991077,Cintya Aguirre,Samara Satélite,Guadalupe Diaz,17/02/2026,celula de credito
9,LCC-017906,Contingencia Crédito,Cosecha Mar 26,595096,x,BRIAN JESUS,brianmurillo281@gmail.com,5644492113,Adrian Gutierrez,Metrópoli Patriotismo,Lilian Juarez,17/02/2026,celula de credito


In [28]:
# Leads activos por asesor iniciales
hca = hist_credito_activos[['espacio automarket','asesor espacio','asesor credito','origen automarket','id lead']]
hca['espacio automarket'] = hca['espacio automarket'].replace(dicc_espacios)
hca['origen automarket'] = hca['origen automarket'].str.strip().str.split(' ').str[0].replace('Apificado','apis').replace('Contingencia','contingencia') # Nos quedamos con la primera palabra, ya sea apificado o contingencia
hca = hca[~hca['asesor espacio'].str.strip().isin(['PRUEBA','prueba','#N/A()','#n/a ()'])]

hca_ases = hca.copy().rename(columns = {'id lead':'leads','asesor espacio':'asesor','espacio automarket':'espacio'})
hca_ases = hca_ases.pivot_table(index=['espacio', 'asesor'], columns = 'origen automarket', aggfunc = {'leads':'nunique'}).reset_index()
hca_ases.columns = hca_ases.columns.map(lambda col: '_'.join([str(x) for x in col if x]))

hca_ascr = hca.copy().rename(columns = {'id lead':'leads','asesor credito':'asesor'})
hca_ascr = hca_ascr.pivot_table(index=['asesor'], columns = 'origen automarket', aggfunc = {'leads':'nunique'}).reset_index()
hca_ascr.columns = hca_ascr.columns.map(lambda col: '_'.join([str(x) for x in col if x]))
hca_ascr['espacio'] = 'celula_credito'

hca = pd.concat([hca_ases, hca_ascr])
hca['asesor'] = hca['asesor'].str.lower().str.strip()
assrs_actvs_leads = assrs_actvs.merge(hca, how = 'left', on = ['asesor','espacio'])
assrs_actvs_leads = assrs_actvs_leads[['espacio','asesor','activo','leads_apis','leads_contingencia']].fillna(0)
assrs_actvs_leads['espacio'] = assrs_actvs_leads['espacio'].replace(dicc_espacios3)

# Leads asignados por asesor

res_asign = salida_leads.copy()
res_asign['asesor espacio'] = res_asign['asesor espacio'].str.lower().str.strip()
res_asign['asesor credito'] = res_asign['asesor credito'].str.lower().str.strip()
res_asign['origen automarket'] = res_asign['origen automarket'].str.strip().str.split(' ').str[0].replace('Apificado','apis').replace('Contingencia','contingencia')

asgns_ases = res_asign.copy().rename(columns = {'espacio automarket':'espacio','asesor espacio':'asesor','id lead':'leads'})
asgns_ases = asgns_ases.pivot_table(index=['espacio','asesor'],columns = 'origen automarket', aggfunc = {'leads':'nunique'}).reset_index()
asgns_ases.columns = asgns_ases.columns.map(lambda col: '_'.join([str(x) for x in col if x]))

asgns_ascr = res_asign.copy().rename(columns = {'espacio automarket':'espacio','asesor credito':'asesor','id lead':'leads'})
asgns_ascr = asgns_ascr.pivot_table(index=['asesor'], columns = 'origen automarket', aggfunc = {'leads':'nunique'}).reset_index()
asgns_ascr.columns = asgns_ascr.columns.map(lambda col: '_'.join([str(x) for x in col if x]))
asgns_ascr['espacio'] = 'celula_credito'

res_asign = pd.concat([asgns_ases, asgns_ascr]).fillna(0)

res_asign = assrs_actvs_leads.merge(res_asign, how = 'left', on = ['espacio','asesor'], suffixes = ['_iniciales','_nuevos'])
res_asign.fillna(0,inplace = True)
try:
  res_asign['leads_apis_finales'] = res_asign['leads_apis_iniciales'] + res_asign['leads_apis_nuevos']
except:
  print(f'No hay asignación nueva de apis')
try:
  res_asign['leads_contingencia_finales'] = res_asign['leads_contingencia_iniciales'] + res_asign['leads_contingencia_nuevos']
except:
    print(f'No hay asignación nueva de contingencia')

res_asign

,espacio,asesor,activo,leads_apis_iniciales,leads_contingencia_iniciales,leads_apis_nuevos,leads_contingencia_nuevos,leads_apis_finales,leads_contingencia_finales
0,Reforma 510,ana karen castro,1.0,2.0,17.0,0.0,1.0,2.0,18.0
1,Reforma 510,betzabe morales,1.0,2.0,20.0,0.0,1.0,2.0,21.0
2,Reforma 510,mariana saldaña,1.0,2.0,19.0,0.0,1.0,2.0,20.0
3,Reforma 510,ximena gutierrez,1.0,4.0,11.0,0.0,2.0,4.0,13.0
4,Reforma 510,yesenia cruz,1.0,3.0,14.0,0.0,1.0,3.0,15.0
5,Samara Satélite,cynthia pacheco,1.0,3.0,22.0,1.0,1.0,4.0,23.0
6,Samara Satélite,erick hernandez,1.0,2.0,20.0,2.0,1.0,4.0,21.0
7,Samara Satélite,guadalupe diaz,1.0,3.0,21.0,1.0,2.0,4.0,23.0
8,Samara Satélite,monica rodriguez,1.0,3.0,12.0,1.0,2.0,4.0,14.0
9,Metrópoli Patriotismo,adonahi rueda,1.0,2.0,14.0,0.0,0.0,2.0,14.0


In [29]:
cols_agg = [c for c in res_asign.columns if all(x != c for x in ['espacio','asesor','activo']) ]
sums = res_asign.groupby('espacio').agg({c: 'sum' for c in cols_agg}).reset_index()
stds = res_asign.groupby('espacio').agg({c: 'std' for c in cols_agg}).reset_index()
sums['asesor'] = 'sum'
stds['asesor'] = 'std'
sums['activo'] = 0
stds['activo'] = 0

res_asign = pd.concat([res_asign,sums,stds]).reset_index(drop=True)
res_asign[cols_agg] = res_asign[cols_agg].round(1)
res_asign

,espacio,asesor,activo,leads_apis_iniciales,leads_contingencia_iniciales,leads_apis_nuevos,leads_contingencia_nuevos,leads_apis_finales,leads_contingencia_finales
0,Reforma 510,ana karen castro,1.0,2.0,17.0,0.0,1.0,2.0,18.0
1,Reforma 510,betzabe morales,1.0,2.0,20.0,0.0,1.0,2.0,21.0
2,Reforma 510,mariana saldaña,1.0,2.0,19.0,0.0,1.0,2.0,20.0
3,Reforma 510,ximena gutierrez,1.0,4.0,11.0,0.0,2.0,4.0,13.0
4,Reforma 510,yesenia cruz,1.0,3.0,14.0,0.0,1.0,3.0,15.0
5,Samara Satélite,cynthia pacheco,1.0,3.0,22.0,1.0,1.0,4.0,23.0
6,Samara Satélite,erick hernandez,1.0,2.0,20.0,2.0,1.0,4.0,21.0
7,Samara Satélite,guadalupe diaz,1.0,3.0,21.0,1.0,2.0,4.0,23.0
8,Samara Satélite,monica rodriguez,1.0,3.0,12.0,1.0,2.0,4.0,14.0
9,Metrópoli Patriotismo,adonahi rueda,1.0,2.0,14.0,0.0,0.0,2.0,14.0


In [30]:
# Celda de validacion tipo assert
id_l_nvos = salida_leads['id lead'].unique()

if not hist[hist['id lead'].isin(id_l_nvos)].shape[0] == 0:
    print(f"{color.RED}Se están duplicando id leads respecto a los que ya existían en la torre de control{color.END}")
    raise SystemExit

## Guardado

In [31]:
# Creamos la hoja de sheets de cero

ahora_dt = datetime.now(ZoneInfo("America/Mexico_City")).strftime('%d-%m-%Y %H:%M')
nb_sheets_salida = f'Salida {ahora_dt}'
nb_hojas = ['asignacion_api_contingencia','resumen_asignaciones']

id_folder_mes_salida = list_file_ids_for_drive_folder(drive, id_drive_salidas)[nb_carpeta_ctes_mes]
create_sheets_in_drive_folder(gc, nb_sheets_salida, id_folder_mes_salida)
id_sheets_salida = listar_archivos(id_folder_mes_salida, mime_types='application/vnd.google-apps.spreadsheet')[nb_sheets_salida]

# Creamos hojas y eliminamos la hoja 1.
# Si esto da error, muy probablemente es porque ya hay un archivo con el mismo nombre en la carpeta. -----> Revisa la carpeta de drive.
crear_hojas_sheets(id_sheets_salida, nb_hojas)
borrar_hojas(id_sheets_salida,['Hoja 1'])

Google Sheet Salida 17-02-2026 18:30 created and updated in folder ID: 1v42XPYCWIaZXjeYdoc2p-jfd6S1K1zBs


['asignacion_api_contingencia', 'resumen_asignaciones'] creadas
['Hoja 1'] eliminada(s)


In [32]:
# Guardamos la salida del proceso de asignación en hojas de respaldo

hoja_df = {'asignacion_api_contingencia': salida_leads, 'resumen_asignaciones':res_asign}
for hoja, df in hoja_df.items():
  update_sheets_in_drive_folder(gc, id_sheets_salida, hoja, df)
  formato_hojas_sheets(id_sheets_salida, [hoja], n_columnas = df.shape[1], letra = 'Source Serif 4' )

print(f'Las salidas se generaron y se guardaron en https://docs.google.com/spreadsheets/d/{id_sheets_salida}')

[attempt 1/3] Google Sheet '1v5GFfjLc9otRFqWMcT0vKjxU4EyLngTTYN3YoWOrYro' - 'asignacion_api_contingencia' updated with new data.
[attempt 1/3] Google Sheet '1v5GFfjLc9otRFqWMcT0vKjxU4EyLngTTYN3YoWOrYro' - 'resumen_asignaciones' updated with new data.
Las salidas se generaron y se guardaron en https://docs.google.com/spreadsheets/d/1v5GFfjLc9otRFqWMcT0vKjxU4EyLngTTYN3YoWOrYro


In [33]:
# Tomamos foto a torre de control
ahora_dt = datetime.now(ZoneInfo("America/Mexico_City")).strftime('%d-%m-%Y %H:%M')
nb_foto = f'FotoTC_{ahora_dt}'
tc2_foto = read_from_google_sheets(gc, id_sheets_tc2, sheetname='asignacion')

# guardamos respaldo
crear_hojas_sheets(id_sheets_salida, [nb_foto])
update_sheets_in_drive_folder(gc, id_sheets_salida, nb_foto, tc2_foto)
formato_hojas_sheets(id_sheets_salida, [nb_foto], n_columnas = tc2_foto.shape[1], letra = 'Source Serif 4' )
# actualizamos si aplica
if actualizar_tc == 'S':
  append_dataframe_to_google_sheet_from_range(gc, id_sheets_tc2, 'asignacion', salida_leads)
else:
  print('No se actualizó la torre de control')

['FotoTC_17-02-2026 18:30'] creadas
[attempt 1/3] Google Sheet '1v5GFfjLc9otRFqWMcT0vKjxU4EyLngTTYN3YoWOrYro' - 'FotoTC_17-02-2026 18:30' updated with new data.
No se actualizó la torre de control
